In [8]:
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from pyproj import Transformer
from shapely.geometry import LineString, Point

In [9]:
namespace = {'k': 'http://www.opengis.net/kml/2.2'}

projection = Transformer.from_crs('EPSG:4326', 'EPSG:32639', always_xy=True)

In [10]:
def read_layer(filename, layer_name, geometry_type):
    with ZipFile(filename) as archive:
        kml_data = archive.read('doc.kml')
    document = ET.fromstring(kml_data)

    for folder in document.findall('.//k:Folder', namespace):
        folder_name = folder.findtext('k:name', namespaces=namespace)
        if folder_name == layer_name:
            selected_folder = folder
            break

    records = []
    for placemark in selected_folder.findall('k:Placemark', namespace):
        name = placemark.findtext('k:name', namespaces=namespace)
        geometry = placemark.find('k:' + geometry_type, namespace)
        text = geometry.findtext('k:coordinates', default='', namespaces=namespace)

        coordinates = []
        for coordinate_text in text.split():
            values = coordinate_text.split(',')
            longitude = float(values[0])
            latitude = float(values[1])
            coordinates.append((longitude, latitude))

        records.append({'name': name, 'coordinates': coordinates})
    return records

In [11]:
lane_records = read_layer('movcud-zolaqlar-velo.kmz', 'Mövcud zolaqlar', 'LineString')
parking_records = read_layer('mpv-v-mps.-son.kmz', 'velosiped məntəqələri yeni', 'Point')

lanes = []
for record in lane_records:
    if len(record['coordinates']) < 2:
        continue

    meter_coordinates = []
    for longitude, latitude in record['coordinates']:
        x, y = projection.transform(longitude, latitude)
        meter_coordinates.append((x, y))

    line = LineString(meter_coordinates)
    lanes.append({'name': record['name'], 'geometry': line})

parking = []
for record in parking_records:
    longitude, latitude = record['coordinates'][0]
    x, y = projection.transform(longitude, latitude)
    parking.append({'name': record['name'], 'longitude': longitude,
                    'latitude': latitude, 'geometry': Point(x, y)})

assert len(parking) == 185

In [12]:
total_meters = 0
street_lengths = {}

for lane in lanes:
    street = lane['name']
    length = lane['geometry'].length
    total_meters += length

    if street not in street_lengths:
        street_lengths[street] = 0
    street_lengths[street] += length

longest_street = ''
longest_length = 0
for street, length in street_lengths.items():
    if length > longest_length:
        longest_street = street
        longest_length = length

total_km = total_meters / 1000
print(f'{total_km:.2f}, {longest_street}')

57.14, 8 Noyabr pr.


In [13]:
nearby_count = 0

for place in parking:
    nearest_distance = float('inf')  # Başlanğıcda sonsuz məsafə.
    for lane in lanes:
        distance = place['geometry'].distance(lane['geometry'])
        if distance < nearest_distance:
            nearest_distance = distance

    if nearest_distance <= 50:
        nearby_count += 1

print(nearby_count)

87


In [14]:
outside_names = []

for place in parking:
    latitude_ok = 40.30 <= place['latitude'] <= 40.46
    longitude_ok = 49.70 <= place['longitude'] <= 50.05
    if not latitude_ok or not longitude_ok:
        outside_names.append(place['name'])

names = ', '.join(outside_names)
print(f'{len(outside_names)}: {names}')

4: MPV-144-2, MPV-145-2, MPV-146-2, MPV-147-2


Azadlıq prospekti üzrə, Vaqif prospektindən şimala yeni zolaq salınmasını təklif edirəm. Şəhərdəki 94 tənha parkinqin 27-sinin ən yaxın zolağı Vaqif prospektindədir. Tikintiyə buradan başlamaq, yaxın parkinqləri qısa bağlantılarla qoşmaq ilk mərhələni sürətləndirə bilər.